# DM-NeRF / DM-SR — study (Colab, Study)
Ative GPU e Internet na interface e execute todas as células. Não edite código.

Autorize o Google Drive quando solicitado. Checkpoints são gravados nele.
O início executa smoke test e piloto de até uma hora. A continuação retoma o treino até o orçamento da sessão.
Uma sessão gratuita não garante convergência. Somente study é executada; nenhuma decisão congelada é alterada.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

def run(*args):
    subprocess.run([str(x) for x in args], check=True)

run('nvidia-smi')
from google.colab import drive
drive.mount('/content/drive')
BASE=Path('/content/dissertacao-dmnerf')
OUTPUT=Path('/content/drive/MyDrive/dissertacao-dmnerf/dmnerf-results')
BASE.mkdir(parents=True,exist_ok=True)
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
REPO=BASE/'sdf-generator'
WORK=BASE/'work'
if not REPO.exists():
    run('git','clone','--branch','dissertacao-2026','--single-branch',
        'https://github.com/Jfsslemos/sdf-generator.git',REPO)
else:
    run('git','-C',REPO,'fetch','origin','dissertacao-2026')
    run('git','-C',REPO,'reset','--hard','origin/dissertacao-2026')
if WORK.exists():
    shutil.rmtree(WORK)
run('git','-C',REPO,'rev-parse','HEAD')
lock=['--lock',OUTPUT/'environment.freeze.txt'] if (OUTPUT/'environment.freeze.txt').exists() else []
run(sys.executable,REPO/'tools/dmnerf/bootstrap.py','--work',WORK,*lock)
PYTHON=WORK/'env/bin/python'
run(PYTHON,'-c','import torch; assert torch.cuda.is_available(), "Ative GPU"; print(torch.cuda.get_device_name(0))')


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/prepare.py','--work',WORK)


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/run.py','--work',WORK,'--output',OUTPUT,'--profile','smoke','--stages','train')


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/run.py','--work',WORK,'--output',OUTPUT,'--profile','pilot','--stages','train')


In [ ]:
print('Resultados:',OUTPUT)
for state in sorted(OUTPUT.rglob('training_state.json')):
    print(state.relative_to(OUTPUT),json.loads(state.read_text()))
print('Pacote:',OUTPUT.parent/(OUTPUT.name+'-bundle.zip'))


## Continuação
Envie o pacote de resultados para revisar tempo/VRAM e decidir viabilidade. No Kaggle, preserve a versão e anexe sua saída ao notebook Resume. No Colab, use Resume com o mesmo Drive.
O smoke é diagnóstico. Avaliação completa e meshing 256³ só ocorrem ao completar o treino. Ausência de superfície é registrada, não corrigida por mudança arbitrária do limiar.
Veja `docs/GPU_RUNBOOK.md`.